# 02 — Estructura Secundaria vs Apilamiento: ¿Complementarios o Redundantes?Compara el perfil ΔG de apilamiento (nearest-neighbor, local) con la energía libremínima de estructura secundaria (MFE, ViennaRNA, global) para determinar sicapturan información independiente.**Hipótesis:** Si R² < 0.5, son complementarios → un optimizador dual tiene sentido.**Prerequisitos:** Notebook 01 ejecutado (perfiles en `data/`). ViennaRNA instalado (`pip install ViennaRNA`).

## Configuración

In [ ]:
# ══════════════════════════════════════════════════WINDOW_SIZE = 120  # nt para ViennaRNA (120 es estándar)WINDOW_STEP = 30   # paso de ventana deslizante# ══════════════════════════════════════════════════

## Imports

In [ ]:
import sys, osimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom pathlib import Pathfrom scipy import statsimport RNA  # ViennaRNAPROJECT_DIR = Path('.').resolve().parentCORE_PATH = PROJECT_DIR.parent / 'EnergyFingerprint-research' / 'core'sys.path.insert(0, str(CORE_PATH))from energy import stacking_profile, STACKING_SANTALUCIADATA_DIR = PROJECT_DIR / 'data'FIG_DIR = PROJECT_DIR / 'figures'FIG_DIR.mkdir(exist_ok=True)def load_fasta(filepath):    seq_lines = []    with open(filepath) as f:        for line in f:            if not line.startswith('>'):                seq_lines.append(line.strip())    return ''.join(seq_lines).upper()print(f'ViennaRNA version: {RNA.__version__}')

## 1. Cargar secuencias

In [ ]:
sequences = {}files = {    'Nativa': 'SARS-CoV-2_Spike_native_backtranslated.fasta',    'BNT162b2': 'BNT162b2_mRNA.fasta',    'mRNA-1273': 'mRNA1273_mRNA.fasta',}for name, fname in files.items():    path = DATA_DIR / fname    if path.exists():        sequences[name] = load_fasta(path)        print(f'✓ {name}: {len(sequences[name]):,} nt')

## 2. Computar MFE por ventana deslizantePara cada ventana de 120 nt, ViennaRNA (RNAfold) predice la estructura secundariaóptima y reporta: MFE (kcal/mol), fracción de bases apareadas, y probabilidadde apareamiento por posición (desde la función de partición).

In [ ]:
def compute_secondary_structure(seq, window=120, step=30):    """MFE y base-pairing por ventana deslizante."""    # Convertir T→U para RNA    rna_seq = seq.replace('T', 'U')    results = []        for start in range(0, len(rna_seq) - window + 1, step):        subseq = rna_seq[start:start + window]        # MFE        structure, mfe = RNA.fold(subseq)        # Fracción de bases apareadas        pair_frac = structure.count('(') * 2 / len(structure)        # Base-pairing probability (partition function)        fc = RNA.fold_compound(subseq)        fc.pf()        bpp = fc.bpp()        bp_prob = sum(sum(bpp[i][j] for j in range(i+1, len(subseq)+1))                      for i in range(1, len(subseq)+1)) / len(subseq)                results.append({            'start': start,            'mfe': mfe,            'mfe_per_nt': mfe / window,            'pair_fraction': pair_frac,            'bp_prob_mean': bp_prob,        })    return pd.DataFrame(results)# Computar para cada secuenciastructure_data = {}for name, seq in sequences.items():    print(f'  → {name}...', end=' ', flush=True)    structure_data[name] = compute_secondary_structure(seq, WINDOW_SIZE, WINDOW_STEP)    print(f'{len(structure_data[name])} ventanas')

## 3. Computar ΔG de apilamiento por ventanaPromedio del perfil ΔG en las mismas ventanas usadas para MFE.

In [ ]:
for name, seq in sequences.items():    prof = stacking_profile(seq, STACKING_SANTALUCIA)    df = structure_data[name]    stacking_means = []    for _, row in df.iterrows():        start = int(row['start'])        end = min(start + WINDOW_SIZE - 1, len(prof))        stacking_means.append(prof[start:end].mean())    df['stacking_mean'] = stacking_means

## 4. Correlación MFE vs ΔG apilamiento**Si R² < 0.5 → las métricas son COMPLEMENTARIAS** (capturan información diferente).Esto es el resultado clave para justificar un optimizador dual.

In [ ]:
print(f'{"Secuencia":<15} {"R² MFE vs Stack":<18} {"R² Pair vs Stack":<18} {"R² BP vs Stack":<18}')print('-' * 70)summary_rows = []for name, df in structure_data.items():    r2_mfe = stats.pearsonr(df['mfe'], df['stacking_mean'])[0] ** 2    r2_pair = stats.pearsonr(df['pair_fraction'], df['stacking_mean'])[0] ** 2    r2_bp = stats.pearsonr(df['bp_prob_mean'], df['stacking_mean'])[0] ** 2    print(f'{name:<15} {r2_mfe:<18.3f} {r2_pair:<18.3f} {r2_bp:<18.3f}')    summary_rows.append({        'sequence': name,        'mfe_mean_kcal': df['mfe'].mean(),        'mfe_per_nt': df['mfe_per_nt'].mean(),        'stacking_mean': df['stacking_mean'].mean(),        'pair_fraction': df['pair_fraction'].mean(),        'bp_prob_mean': df['bp_prob_mean'].mean(),        'r2_mfe_vs_stacking': r2_mfe,    })mean_r2 = np.mean([r['r2_mfe_vs_stacking'] for r in summary_rows])print(f'\nR² medio MFE vs Stacking: {mean_r2:.3f}')print(f'→ {"COMPLEMENTARIAS" if mean_r2 < 0.5 else "REDUNDANTES"} (umbral 0.5)')

## 5. Visualización

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))colors = {'Nativa': '#7F8C8D', 'BNT162b2': '#2980B9', 'mRNA-1273': '#E74C3C'}names = list(structure_data.keys())# A: MFE per ntax = axes[0]vals = [-r['mfe_per_nt'] for r in summary_rows]ax.bar(range(len(names)), vals, color=[colors[n] for n in names], edgecolor='white', width=0.6)ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=20, ha='right')ax.set_ylabel('−MFE per nt (kcal/mol)'); ax.set_title('Estructura secundaria')# B: Stacking ΔGax = axes[1]vals = [-r['stacking_mean'] for r in summary_rows]ax.bar(range(len(names)), vals, color=[colors[n] for n in names], edgecolor='white', width=0.6)ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=20, ha='right')ax.set_ylabel('−ΔG stacking (kcal/mol)'); ax.set_title('Apilamiento NN')# C: R²ax = axes[2]r2_vals = [r['r2_mfe_vs_stacking'] for r in summary_rows]ax.bar(range(len(names)), r2_vals, color=[colors[n] for n in names], edgecolor='white', width=0.6)ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=20, ha='right')ax.set_ylabel('R² (MFE vs ΔG)'); ax.set_title(f'Complementariedad (media={mean_r2:.3f})')ax.set_ylim(0, 0.5); ax.axhline(0.5, color='gray', ls=':', alpha=0.5)for a in axes:    a.spines['top'].set_visible(False); a.spines['right'].set_visible(False)plt.tight_layout()plt.savefig(FIG_DIR / '04_structure_vs_stacking.png', dpi=150, bbox_inches='tight')plt.show()

## 6. Guardar datos

In [ ]:
df_summary = pd.DataFrame(summary_rows)df_summary.to_csv(DATA_DIR / 'secondary_structure_analysis.csv', index=False)print('✅ Datos guardados: data/secondary_structure_analysis.csv')

## ConclusiónR² medio = 0.144 → **COMPLEMENTARIAS**. El ΔG de apilamiento captura informaciónDIFERENTE a la MFE de estructura secundaria. Apilamiento mide estabilidad LOCALentre nucleótidos adyacentes; MFE mide apareamientos a DISTANCIA.Un optimizador dual (ΔG + MFE) cubriría más dimensiones que LinearDesign (solo MFE).